# Notebook 3 — Convergent Residues and Hypothesis Testing

## Learning objectives
- Compute **per-site conservation** (Shannon entropy) and compare between groups
- Detect **convergent amino acid residues** shared by echolocating species
- Build a **permutation test from scratch** and understand the null hypothesis
- Compare convergent site rates between **prestin and control subfamilies**

In [ ]:
import os, re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from Bio import AlignIO, SeqIO

DATA = os.path.join("..", "data")
SUB_DIR = os.path.join(DATA, "subfamilies")
FIGS = os.path.join("..", "figures")
AA = set("ACDEFGHIKLMNPQRSTVWY")

---
## 1. Load the prestin alignment and classify species

You need to define which taxids correspond to echolocating species.
Copy this from Notebook 2, or fill it in from your species list.

In [ ]:
# FILL IN from your actual data (Notebook 1 species list)
ECHOLOCATING_TAXIDS = {
    # Toothed whales
    "9739", "9749", "9755",
    # Echolocating bats — FILL IN your bat taxids here
    # "59479", "9407", ...
}

# Find prestin trimmed alignment
prestin_alns = sorted([f for f in os.listdir(SUB_DIR)
                       if "prestin" in f and f.endswith((".gt01", ".gt01.fa"))])
if not prestin_alns:
    # Try any trimmed file
    prestin_alns = sorted([f for f in os.listdir(SUB_DIR) if "prestin" in f and "gt" in f])

print(f"Available prestin alignments: {prestin_alns}")
PRESTIN_ALN = os.path.join(SUB_DIR, prestin_alns[0]) if prestin_alns else None

if PRESTIN_ALN:
    aln = AlignIO.read(PRESTIN_ALN, "fasta")
    print(f"Loaded: {len(aln)} sequences × {aln.get_alignment_length()} columns")

    # Classify
    echo_idx = [i for i in range(len(aln)) if aln[i].id.split(".")[0] in ECHOLOCATING_TAXIDS]
    nonecho_idx = [i for i in range(len(aln)) if i not in echo_idx]
    print(f"Echolocators: {len(echo_idx)}, Non-echolocators: {len(nonecho_idx)}")

    if len(echo_idx) == 0:
        print("\n⚠ No echolocators found! You need to fill in ECHOLOCATING_TAXIDS above.")
        print("Look at the taxids from Notebook 1 species list.")

---
## 2. Per-site conservation: Shannon entropy

$$H = -\sum_{a \in \text{amino acids}} p_a \log_2(p_a)$$

$H = 0$ means perfectly conserved. $H \approx 4.3$ means maximum variability.

In [ ]:
def shannon_entropy(column):
    """Shannon entropy of an alignment column."""
    residues = [c for c in column if c in AA]
    if not residues:
        return 0.0
    counts = Counter(residues)
    n = len(residues)
    return -sum((c/n) * np.log2(c/n) for c in counts.values())

def conservation_profile(aln, indices):
    """Per-site entropy for a subset of sequences."""
    return np.array([
        shannon_entropy([str(aln[i].seq[pos]) for i in indices])
        for pos in range(aln.get_alignment_length())
    ])

H_echo = conservation_profile(aln, echo_idx)
H_nonecho = conservation_profile(aln, nonecho_idx)
H_all = conservation_profile(aln, list(range(len(aln))))

print(f"Mean entropy — echolocators: {H_echo.mean():.3f}, non-echo: {H_nonecho.mean():.3f}, all: {H_all.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
positions = np.arange(aln.get_alignment_length())

axes[0].bar(positions, H_echo, color="#D85A30", alpha=0.7, width=1)
axes[0].set_ylabel("Entropy\n(echolocators)")
axes[0].set_title("Per-site conservation — prestin (SLC26A5)", fontweight="bold")

axes[1].bar(positions, H_nonecho, color="#85B7EB", alpha=0.7, width=1)
axes[1].set_ylabel("Entropy\n(non-echolocators)")

diff = H_nonecho - H_echo
colors = ["#D85A30" if d > 0.3 else "#85B7EB" if d < -0.3 else "#CCCCCC" for d in diff]
axes[2].bar(positions, diff, color=colors, alpha=0.8, width=1)
axes[2].axhline(0, color="black", linewidth=0.5)
axes[2].set_ylabel("Differential\n(nonecho − echo)")
axes[2].set_xlabel("Alignment position")

plt.tight_layout()
plt.savefig(os.path.join(FIGS, "03_conservation.png"), dpi=150, bbox_inches="tight")
plt.show()

### ✏️ Exercise 3.1
Red bars in the bottom panel = positions more conserved in echolocators.
Why might echolocators show **lower entropy** at specific sites?
What kind of selection would produce this pattern?

---
## 3. Detecting convergent residues

A **convergent site** is a position where:
1. ≥60% of echolocators share the same amino acid
2. That amino acid differs from the non-echolocator consensus
3. The amino acid is rare in non-echolocators (high specificity)

In [ ]:
def find_convergent_sites(aln, echo_idx, nonecho_idx,
                           min_echo_frac=0.6, min_echo_count=3):
    """Find positions where echolocators share an AA different from non-echolocators."""
    sites = []
    for pos in range(aln.get_alignment_length()):
        echo_aas = [str(aln[i].seq[pos]) for i in echo_idx if str(aln[i].seq[pos]) in AA]
        nonecho_aas = [str(aln[i].seq[pos]) for i in nonecho_idx if str(aln[i].seq[pos]) in AA]

        if len(echo_aas) < min_echo_count or len(nonecho_aas) < 3:
            continue

        echo_counts = Counter(echo_aas)
        top_aa, top_count = echo_counts.most_common(1)[0]
        echo_frac = top_count / len(echo_aas)
        if echo_frac < min_echo_frac:
            continue

        nonecho_top = Counter(nonecho_aas).most_common(1)[0][0]
        if top_aa != nonecho_top:
            nonecho_has = Counter(nonecho_aas).get(top_aa, 0) / len(nonecho_aas)
            sites.append({
                "position": pos,
                "echo_aa": top_aa,
                "echo_fraction": echo_frac,
                "nonecho_consensus": nonecho_top,
                "nonecho_has_echo_aa": nonecho_has,
                "specificity": echo_frac - nonecho_has,
            })
    return sites

conv_sites = find_convergent_sites(aln, echo_idx, nonecho_idx)
print(f"Convergent sites: {len(conv_sites)} / {aln.get_alignment_length()} columns")
print(f"Rate: {len(conv_sites)/aln.get_alignment_length():.4f}\n")

print(f"{'Pos':>5s}  {'EchoAA':>6s}  {'Frac':>6s}  {'NonEcho':>7s}  {'Specif':>7s}")
print("-" * 40)
for s in sorted(conv_sites, key=lambda x: -x["specificity"])[:15]:
    print(f"{s['position']:>5d}  {s['echo_aa']:>6s}  {s['echo_fraction']:>6.0%}  "
          f"{s['nonecho_consensus']:>7s}  {s['specificity']:>7.3f}")

### ✏️ Exercise 3.2
Re-run with `min_echo_frac` set to 0.5, 0.7, and 0.8.
How does the count change? What is the sensitivity/specificity trade-off?

---
## 4. Permutation test

### The question
Does prestin have more convergent sites than expected if the
"echolocator" label were randomly assigned?

### The method
1. Count observed convergent sites ✓
2. **Shuffle** echolocator/non-echolocator labels randomly
3. Recount convergent sites with shuffled labels
4. Repeat 1000 times → null distribution
5. p-value = fraction of permutations ≥ observed

In [ ]:
def permutation_test(aln, echo_idx, nonecho_idx,
                      n_perm=1000, min_echo_frac=0.6, seed=42):
    """Label-shuffling permutation test for convergent site count."""
    rng = np.random.default_rng(seed)
    n_echo = len(echo_idx)
    all_idx = list(range(len(aln)))

    observed = len(find_convergent_sites(aln, echo_idx, nonecho_idx,
                                          min_echo_frac=min_echo_frac))

    null = np.zeros(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(all_idx)
        null[i] = len(find_convergent_sites(aln, list(perm[:n_echo]),
                                              list(perm[n_echo:]),
                                              min_echo_frac=min_echo_frac))
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{n_perm}...")

    p = np.mean(null >= observed)
    z = (observed - null.mean()) / null.std() if null.std() > 0 else 0
    return {"observed": observed, "null": null, "p_value": p, "z_score": z,
            "null_mean": null.mean(), "null_std": null.std()}

In [ ]:
print("Running permutation test (may take a few minutes)...\n")
result = permutation_test(aln, echo_idx, nonecho_idx, n_perm=1000)

print(f"Observed: {result['observed']} convergent sites")
print(f"Null:     {result['null_mean']:.1f} ± {result['null_std']:.2f}")
print(f"z-score:  {result['z_score']:.3f}")
print(f"p-value:  {result['p_value']:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(result["null"], bins=40, color="#85B7EB", alpha=0.7, edgecolor="white",
        density=True, label="Null (shuffled labels)")
ax.axvline(result["observed"], color="#D85A30", linewidth=2.5, linestyle="--",
           label=f"Observed (prestin)")
ax.set_xlabel("Number of convergent sites")
ax.set_ylabel("Density")
ax.set_title(f"Permutation test — p = {result['p_value']:.4f}, z = {result['z_score']:.2f}",
             fontweight="bold")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "03_permtest.png"), dpi=150, bbox_inches="tight")
plt.show()

### ✏️ Exercise 3.3
1. What does p < 0.05 mean here, **in plain English**?
2. Why do we shuffle **labels**, not sequences or columns?
3. Re-run with 5000 permutations. Does the p-value stabilize?
4. What would it mean if the control subfamily also had a low p-value?

---
## 5. Cross-subfamily comparison

### ✏️ Exercise 3.4
Run `find_convergent_sites` on your control subfamily (from Notebook 2).
Compare its convergent site rate to prestin's.
Is prestin enriched? By how much?

In [ ]:
# YOUR CODE: load control alignment, classify echo/nonecho, find convergent sites
# ctrl_aln = AlignIO.read(os.path.join(SUB_DIR, "control.clustalo_fasttree.gt01"), "fasta")
# ctrl_echo_idx = [i for i in range(len(ctrl_aln)) if ctrl_aln[i].id.split(".")[0] in ECHOLOCATING_TAXIDS]
# ...

---
## 6. Mark convergent sites on the conservation plot

In [ ]:
conv_positions = [s["position"] for s in conv_sites]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(positions, H_all, color="#CCCCCC", width=1, alpha=0.5, label="Entropy (all species)")
for cp in conv_positions:
    ax.axvline(cp, color="#D85A30", linewidth=1.2, alpha=0.6)
ax.set_xlabel("Alignment position")
ax.set_ylabel("Shannon entropy")
ax.set_title(f"Convergent sites (red) on prestin conservation ({len(conv_positions)} sites)",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "03_convergent_map.png"), dpi=150, bbox_inches="tight")
plt.show()

### ✏️ Exercise 3.5
Do convergent sites tend to fall in **conserved** or **variable** regions?
Compute the mean entropy at convergent positions vs all positions.
What does this tell you about the nature of these substitutions?

In [ ]:
# YOUR CODE
# H_conv = H_all[conv_positions]
# print(f"Mean entropy at convergent sites: {H_conv.mean():.3f}")
# print(f"Mean entropy overall: {H_all.mean():.3f}")

---
## Summary

- **Shannon entropy** reveals sites under different conservation regimes
  in echolocators vs non-echolocators
- **Convergent residues** are positions where echolocators independently
  evolved the same amino acid
- The **permutation test** shows prestin has significantly more convergence
  than expected by chance
- In **Notebook 4**, we map these sites onto the 3D structure.